# BirdClef 2026 - Inference Notebook

This notebook loads the trained model and generates predictions for submission.

To push notebook to kaggle:
kaggle kernels push -p notebooks/

     It uses the kernel-metadata.json in that directory to determine which notebook file to push and what to name
     it.

In [ ]:
# Install dependencies
!pip install -q torch librosa soundfile pandas numpy torchvision tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm
import gc

## Setup

In [ ]:
# Configuration
SAMPLE_RATE = 32000
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
DURATION = 5
NUM_CLASSES = 234

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Model Definition

In [ ]:
class BirdClefModel(nn.Module):
    def __init__(self, num_classes=234):
        super().__init__()
        self.backbone = models.efficientnet_b2(weights=None)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        x = self.backbone(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x

## Load Model and Data

In [ ]:
# Load the trained model
model = BirdClefModel(num_classes=NUM_CLASSES)
checkpoint = torch.load('/kaggle/input/datasets/krist0phersmith/birdclef2026-model/best_model.pt', 
                        map_location=device,
                        weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()
print("Model loaded successfully!")
print(f"Val Loss: {checkpoint.get('val_loss', 'N/A'):.4f}, Val Acc: {checkpoint.get('val_acc', 'N/A'):.4f}")

In [ ]:
# Load taxonomy to get class names
taxonomy = pd.read_csv('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
label_cols = taxonomy['primary_label'].tolist()
print(f"Number of classes: {len(label_cols)}")

In [ ]:
# Load sample submission to get expected format
sample_sub = pd.read_csv('/kaggle/input/competitions/birdclef-2026/sample_submission.csv')
submission_label_cols = [c for c in sample_sub.columns if c != 'row_id']
print(f"Submission columns: {len(submission_label_cols)}")
print(sample_sub.head())

## Audio Processing

In [ ]:
def compute_spectrogram(y, sr):
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_db.astype(np.float32)

def load_and_process_audio(audio_path):
    """Process audio file into 5-second segments."""
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    
    segment_samples = SAMPLE_RATE * DURATION
    spectrograms = []
    row_ids = []
    
    for start in range(0, len(y), segment_samples):
        end = min(start + segment_samples, len(y))
        segment = y[start:end]
        
        if len(segment) < segment_samples:
            segment = np.pad(segment, (0, segment_samples - len(segment)))
        
        spec = compute_spectrogram(segment, sr)
        spectrograms.append(spec)
        
        filename = os.path.basename(audio_path).replace('.ogg', '')
        start_sec = start // SAMPLE_RATE
        row_id = f"{filename}_{start_sec + DURATION}"
        row_ids.append(row_id)
    
    return spectrograms, row_ids

## Get Test Files

In [ ]:
# Find test audio files
test_dir = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'

if os.path.exists(test_dir):
    test_files = [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.ogg')]
    print(f"Found {len(test_files)} test files")
else:
    test_files = []
    print("Test directory not found — no test files available (expected during non-submission runs).")


In [ ]:
# Process all test files
all_predictions = []
all_row_ids = []

for audio_path in tqdm(test_files, desc="Processing"):
    spectrograms, row_ids = load_and_process_audio(audio_path)
    
    if not spectrograms:
        continue
    
    # Convert to tensor
    specs_tensor = torch.tensor(np.array(spectrograms)).unsqueeze(1)
    
    # Make predictions in batches
    batch_size = 16
    predictions = []
    
    for i in range(0, len(specs_tensor), batch_size):
        batch = specs_tensor[i:i+batch_size].to(device)
        with torch.no_grad():
            outputs = model(batch)
            probs = torch.sigmoid(outputs)
        predictions.append(probs.cpu().numpy())
    
    predictions = np.concatenate(predictions, axis=0)
    all_predictions.append(predictions)
    all_row_ids.extend(row_ids)

print(f"Total segments: {len(all_row_ids)}")

## Create Submission

In [ ]:
# Combine predictions and build submission
if all_predictions:
    print(f"Building submission from {len(all_row_ids)} predicted segments...")
    predictions_array = np.concatenate(all_predictions, axis=0)

    submission = pd.DataFrame({'row_id': all_row_ids})
    label_to_idx = {label: i for i, label in enumerate(label_cols)}
    for col in submission_label_cols:
        model_idx = label_to_idx[col]
        submission[col] = predictions_array[:, model_idx]

    # Left-join onto sample_sub so row order and any missing rows are handled correctly
    final_submission = sample_sub[['row_id']].merge(submission, on='row_id', how='left')
    final_submission[submission_label_cols] = final_submission[submission_label_cols].fillna(1 / NUM_CLASSES)
    print(f"Submission shape: {final_submission.shape}")

else:
    # No test files found — fall back to sample submission defaults.
    # This is expected when running outside a Kaggle submission environment
    # (test soundscapes are hidden until submission time).
    print("No test files found — using sample submission defaults (non-submission run).")
    final_submission = sample_sub.copy()
    print(f"Submission shape: {final_submission.shape}")

final_submission.head()


In [ ]:
# Sanity check: verify all expected row_ids are present and no NaNs
expected_row_ids = set(sample_sub['row_id'].tolist())
output_row_ids = set(final_submission['row_id'].tolist())

missing = expected_row_ids - output_row_ids
extra = output_row_ids - expected_row_ids
nan_count = final_submission[submission_label_cols].isna().sum().sum()

print(f"Expected row_ids : {len(expected_row_ids)}")
print(f"Output row_ids   : {len(output_row_ids)}")
print(f"Missing row_ids  : {len(missing)}")
print(f"Extra row_ids    : {len(extra)}")
print(f"NaN values       : {nan_count}")
print("All checks passed!" if not missing and not extra and nan_count == 0 else "WARNING: issues found above.")


In [ ]:
# Save submission
final_submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")

# Verify
sub_check = pd.read_csv('submission.csv')
print(f"Shape: {sub_check.shape}")
print(sub_check.head())